# LELA60341: Research Methods in Computational and Corpus Linguistics Week 10

## A text classifier: putting the classes together

In Week 8 you wrote `LogisticRegression`, which learns from rows of numbers, and in Week 9 `TextEncoder`, which turns texts into rows. This week you put them together into a `TextLogisticRegression` that can be trained and evaluated directly on texts: the Yelp review classifier from last week's LELA60331 seminar (*Training and evaluating a text classifier*), as one object.

```python
model = TextLogisticRegression(TextEncoder(5000), lr=0.4, n_epochs=500)
model.train(train_texts, train_labels)
print(model.evaluate(test_texts, test_labels))
```

| Part | Topic |
|---|---|
| 0 | Set-up: the data, your classes from Weeks 8 and 9, and some helper functions |
| 1 | Putting the classes together: "is a" and "has a" |
| 2 | Training and evaluating on the Yelp reviews |
| 3 | Inspecting the weights |
| 4 | A better tokeniser: just swap the encoder |

Read each explanation, run the example cells, then try the 🏋️ exercises. In an exercise, replace each `???` with your own code, then run the ✅ check cell underneath. When you see a 🔮 **Predict** prompt, decide what you think will happen *before* you run the cell. Parts marked ⭐ **Challenge** are optional.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used by every exercise.
def check(label, condition):
    """Print a tick or a cross instead of crashing, so you can keep working."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 0 · Set-up

Run the next four cells. They download the reviews, and set up:

* the data (`load_reviews` and the training and test split) and the `TextEncoder` class, from Week 9
* the `LogisticRegression` class from Week 8, with its helper functions `dot`, `mat_vec` and `transpose`
* three more helper functions, from the LELA60331 seminars:
  * `sparse_mat_vec(M, vector)`: a score for every row of a sparse matrix (LELA60331 Week 8, Activity 8)
  * `sparse_transpose_mat_vec(M, vector, num_columns)`: a gradient for every column (LELA60331 Week 8, Activity 9)
  * `sigmoid(z)`: the safer sigmoid from LELA60331 Week 9, which doesn't fail for very large negative scores. It replaces the simple version from Week 8, and gives exactly the same values.

In [ ]:
!wget https://raw.githubusercontent.com/cbannard/lela60331_24-25/refs/heads/main/data/yelp_reviews.txt

In [ ]:
import math
import random
import matplotlib.pyplot as plt

def load_reviews(path):
    texts = []
    labels = []
    with open(path) as fp:
        lines = fp.readlines()
    for line in lines[1:]:              # skip the header line
        fields = line.rstrip().split("\t")
        texts.append(fields[0])
        labels.append(int(fields[1] == "positive"))
    return texts, labels

texts, labels = load_reviews("yelp_reviews.txt")

random.seed(1)                          # the same split as in Week 9 and in LELA60331
train_ints = random.sample(range(len(texts)), 8000)
train_set = set(train_ints)
test_ints = [i for i in range(len(texts)) if i not in train_set]
train_texts = [texts[i] for i in train_ints]
train_labels = [labels[i] for i in train_ints]
test_texts = [texts[i] for i in test_ints]
test_labels = [labels[i] for i in test_ints]


class TextEncoder:
    """Turn texts into sparse one-hot rows over a vocabulary of the most frequent words."""

    def __init__(self, vocab_size=5000):
        self._vocab_size = vocab_size
        self._type_list = []
        self._word_to_index = {}

    def tokenise(self, text):
        """Split a text into a list of tokens."""
        return text.split()

    def fit_vocabulary(self, texts):
        """Build the vocabulary from the most frequent tokens in texts."""
        counts = {}
        for text in texts:
            for word in self.tokenise(text):
                counts[word] = counts.get(word, 0) + 1
        pairs = sorted([(count, word) for word, count in counts.items()], reverse=True)
        self._type_list = [word for count, word in pairs[:self._vocab_size]]
        self._word_to_index = {}
        for j in range(len(self._type_list)):
            self._word_to_index[self._type_list[j]] = j

    def __len__(self):
        return len(self._type_list)

    def __getitem__(self, j):
        return self._type_list[j]

    def index(self, word):
        """The column of word, or None if it is not in the vocabulary."""
        return self._word_to_index.get(word)

    def one_hot_encode(self, texts):
        """A list with one sorted list of columns per text."""
        M = []
        for text in texts:
            row = set()
            for word in self.tokenise(text):
                if word in self._word_to_index:
                    row.add(self._word_to_index[word])
            M.append(sorted(row))
        return M

    def decode(self, row):
        """The words for the columns in a sparse row."""
        return [self[j] for j in row]

print(len(train_texts), "training and", len(test_texts), "test reviews")

In [ ]:
def dot(a, b):
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

def mat_vec(matrix, vector):
    return [dot(row, vector) for row in matrix]

def transpose(matrix):
    return [[row[j] for row in matrix] for j in range(len(matrix[0]))]

class LogisticRegression:
    """Logistic regression for examples given as rows of numeric features."""

    def __init__(self, lr=0.5, n_epochs=1000, seed=10):
        self._lr = lr
        self._n_epochs = n_epochs
        self._seed = seed
        self._weights = []
        self._bias = 0
        self._loss_history = []

    # ---------------------------------------------------------------- steps that depend on how the features are stored
    def _num_features(self, X):
        return len(X[0])

    def _start(self, num_features):
        random.seed(self._seed)
        self._weights = [random.random() for j in range(num_features)]
        self._bias = 0

    def _scores(self, X):
        return mat_vec(X, self._weights)

    def _gradients(self, X, errors):
        return mat_vec(transpose(X), errors)

    # ---------------------------------------------------------------- prediction
    def predict_proba(self, X):
        """The probability that each example is positive."""
        return [sigmoid(value + self._bias) for value in self._scores(X)]

    def predict(self, X):
        """The predicted class (1 or 0) of each example."""
        return [int(p > 0.5) for p in self.predict_proba(X)]

    # ---------------------------------------------------------------- training
    def average_loss(self, q, y):
        eps = 0.00001          # a tiny number, so that we never take the log of 0
        losses = []
        for k in range(len(y)):
            if y[k] == 1:
                losses.append(-math.log2(q[k] + eps))
            else:
                losses.append(-math.log2(1 - q[k] + eps))
        return sum(losses) / len(losses)

    def train(self, X, y, verbose=False):
        """Learn the weights and bias from the rows X and their labels y (1 or 0)."""
        num_samples = len(y)
        self._start(self._num_features(X))
        self._loss_history = []
        for epoch in range(self._n_epochs):
            q = [sigmoid(value + self._bias) for value in self._scores(X)]    # as in predict_proba
            self._loss_history.append(self.average_loss(q, y))
            errors = [q[k] - y[k] for k in range(num_samples)]
            dw = [value / num_samples for value in self._gradients(X, errors)]
            db = sum(errors) / num_samples
            self._weights = [self._weights[j] - self._lr * dw[j] for j in range(len(self._weights))]
            self._bias = self._bias - self._lr * db
            if verbose and epoch % 50 == 0:
                print("epoch", epoch, " loss", round(self._loss_history[-1], 3))

    def loss_history(self):
        return self._loss_history

    def parameters(self):
        return self._weights, self._bias

    # ---------------------------------------------------------------- evaluation
    def evaluate(self, X, y):
        """A dict with the accuracy, precision and recall of the model on X."""
        predicted = self.predict(X)
        correct = 0
        tp = 0
        fp = 0
        fn = 0
        for k in range(len(y)):
            if predicted[k] == y[k]:
                correct += 1
            if predicted[k] == 1 and y[k] == 1:
                tp += 1
            if predicted[k] == 1 and y[k] == 0:
                fp += 1
            if predicted[k] == 0 and y[k] == 1:
                fn += 1
        return {"accuracy": correct / len(y),
                "precision": tp / (tp + fp),
                "recall": tp / (tp + fn)}

In [ ]:
def sigmoid(z):
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    else:
        return math.exp(z) / (1 + math.exp(z))      # the same value, without calculating math.exp(-z)

def sparse_mat_vec(M, vector):
    result = []
    for row in M:
        total = 0
        for j in row:
            total += vector[j]
        result.append(total)
    return result

def sparse_transpose_mat_vec(M, vector, num_columns):
    result = [0] * num_columns
    for i in range(len(M)):
        for j in M[i]:
            result[j] += vector[i]
    return result

---
# Part 1 · Putting the classes together: "is a" and "has a"

How should the two classes be combined? There are two kinds of relationship between classes:

* **"is a"**: a text classifier *is a* logistic regression model. It trains, predicts and evaluates in exactly the same way; only the form of its input is different. This is **inheritance** (Week 7): `TextLogisticRegression` is a **subclass** of `LogisticRegression`, and inherits `train`, `predict`, `evaluate` and the rest.
* **"has a"**: a text classifier *has a* text encoder, which it uses to turn its texts into rows. It isn't itself an encoder. This is called **composition**: one object is stored in an **instance variable** of another, here `self._encoder`, and the outer object passes some of its work on to it.

Composition has a big advantage here: the classifier doesn't care **which** encoder it has, as long as the encoder has the right methods. To try a different tokeniser, you just give the classifier a different encoder (Part 4).

### What changes in the subclass

Here is where Week 8's design pays off. `LogisticRegression` keeps everything that depends on how the features are stored in four small methods. Rows of numbers there; sparse one-hot rows here. So the subclass overrides just those four:

| Method | `LogisticRegression` (Week 8) | `TextLogisticRegression` |
|---|---|---|
| `_num_features(X)` | the length of a row | the size of the encoder's vocabulary |
| `_start(num_features)` | random weights, bias 0 | the same, then a random bias (as in LELA60331 Week 9) |
| `_scores(X)` | `mat_vec` with the weights | `sparse_mat_vec` with the weights |
| `_gradients(X, errors)` | `mat_vec` with the transpose | `sparse_transpose_mat_vec`, with one column per vocabulary word |

The user of the class should be able to give it **texts**, so two public methods turn the texts into one-hot rows, then hand them to the parent's version with `super()`:

* `predict_proba(texts)`: one-hot encode the texts, then call the parent's `predict_proba` with the rows
* `train(texts, labels)`: build the vocabulary if the encoder doesn't have one yet, then call the parent's `train` with the one-hot rows

`predict` and `evaluate` don't need to change at all. Both call `self.predict_proba`, which now means the subclass's version, so they work with texts automatically. (This is why Week 8's `train` doesn't call `predict_proba`: inside `train` the data has already been turned into rows.)

The subclass also passes two jobs on to its encoder (`fit_vocabulary` and `one_hot_encode`) and adds a new method, `top_words(n)`: the `n` words with the lowest weights and the `n` with the highest, as in LELA60331 Week 9, Activity 7. Each weight belongs to one word: `self._encoder[j]` is the word for weight `j`.

### 🏋️ Exercise 1 · Practice — Complete `TextLogisticRegression`

Fill in the blanks 1–7. Every blank is a single line. The cell won't run until every `???` is replaced, so work through them in order. The tests underneath train a model on a tiny corpus of eight made-up reviews, which takes no time at all.

In [ ]:
class TextLogisticRegression(LogisticRegression):
    """Logistic regression for texts: a LogisticRegression that has a TextEncoder."""

    def __init__(self, encoder, lr=0.4, n_epochs=500, seed=2):
        ???                     # 1. the parent's instance variables
        self._encoder = ???                                  #    "has a" TextEncoder

    # ---------------------------------------------------------------- jobs passed on to the encoder
    def fit_vocabulary(self, texts):
        ???                      # 2.

    def one_hot_encode(self, texts):
        return ???

    # ---------------------------------------------------------------- the features are now sparse rows
    def _num_features(self, M):
        return ???                                # 3.

    def _start(self, num_features):
        super()._start(num_features)            # random weights, as in Week 8
        self._bias = random.random()            # LELA60331 Week 9 also starts the bias at random

    def _scores(self, M):
        return ???                  # 4.

    def _gradients(self, M, errors):
        return ???

    # ---------------------------------------------------------------- texts in, rows inside
    def predict_proba(self, texts):
        return ???            # 5.

    def train(self, texts, labels, verbose=False):
        if len(self._encoder) == 0:             # no vocabulary yet
            self.fit_vocabulary(texts)
        ???          # 6.

    # ---------------------------------------------------------------- new
    def top_words(self, n):
        """The n words with the lowest weights, and the n with the highest (highest first)."""
        pairs = ???   # 7.
        lowest = [word for weight, word in pairs[:n]]
        highest = [word for weight, word in pairs[::-1][:n]]
        return lowest, highest


tiny_texts = ["great food", "great service", "lovely food great staff", "lovely place",
              "awful food", "awful service", "rude staff awful place", "rude service"]
tiny_labels = [1, 1, 1, 1, 0, 0, 0, 0]

tiny_model = TextLogisticRegression(TextEncoder(10), lr=0.5, n_epochs=200)
tiny_model.train(tiny_texts, tiny_labels)
print(tiny_model.top_words(2))
print([round(p, 3) for p in tiny_model.predict_proba(["great place", "awful staff"])])
print(tiny_model.predict(tiny_texts))
print(tiny_model.evaluate(tiny_texts, tiny_labels))
# you should get: (['awful', 'rude'], ['great', 'lovely']), then [0.99, 0.034], then [1, 1, 1, 1, 0, 0, 0, 0], then all three 1.0

In [ ]:
# ✅ Check
check("1. is a LogisticRegression", lambda: isinstance(tiny_model, LogisticRegression))
check("1. has a TextEncoder", lambda: isinstance(tiny_model._encoder, TextEncoder))
check("2. the vocabulary", lambda: len(tiny_model._encoder) == 8 and tiny_model.one_hot_encode(["great great food"]) == [sorted([tiny_model._encoder.index("great"), tiny_model._encoder.index("food")])])
check("3. one weight per vocabulary word", lambda: tiny_model._num_features(None) == 8 and len(tiny_model.parameters()[0]) == 8)
check("4. _scores", lambda: tiny_model._scores([[0, 2], []]) == [tiny_model.parameters()[0][0] + tiny_model.parameters()[0][2], 0])
check("4. _gradients", lambda: tiny_model._gradients([[0, 2], [1, 2]], [10, 1]) == [10, 1, 11, 0, 0, 0, 0, 0])
check("5. predict_proba takes texts", lambda: tiny_model.predict(["great place", "awful staff"]) == [1, 0])
check("6. the loss went down", lambda: len(tiny_model.loss_history()) == 200 and tiny_model.loss_history()[-1] < tiny_model.loss_history()[0])
check("evaluate works on texts, unchanged", lambda: tiny_model.evaluate(tiny_texts, tiny_labels) == {"accuracy": 1.0, "precision": 1.0, "recall": 1.0})
check("7. top_words", lambda: tiny_model.top_words(2) == (["awful", "rude"], ["great", "lovely"]))

🔮 **Predict:** when you call `tiny_model.evaluate(tiny_texts, tiny_labels)`, which class's version of each of these methods runs: `evaluate`, `predict`, `predict_proba`, `_scores`? Check your answer by reading the code.

---
# Part 2 · Training and evaluating on the Yelp reviews

### 🏋️ Exercise 2 · Practice — The LELA60331 classifier, as an object

1. Make `model`, a `TextLogisticRegression` with a new `TextEncoder` of 5000 words, a learning rate of 0.4 and 500 epochs.
2. Build its vocabulary from **all** the reviews in `texts`, as the LELA60331 seminar did, *before* training.
3. Train it on the training texts and labels, with `verbose=True` to see its progress. Training takes about a minute.
4. `test_results` and `train_results`: the results of evaluating it on the test set and on the training set

In [ ]:
model = ???    # 1.
???                                               # 2.
???                      # 3.

plt.plot(model.loss_history())
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()

test_results = ???                    # 4.
train_results = ???
print("test:    ", test_results)
print("training:", train_results)

In [ ]:
# ✅ Check
check("1. 5000 weights", lambda: len(model.parameters()[0]) == 5000)
check("3. the same final loss as LELA60331 Week 9 (0.598)", lambda: abs(model.loss_history()[-1] - 0.598) < 0.001)
check("4. test accuracy 0.82, precision 0.807, recall 0.829",
      lambda: abs(test_results["accuracy"] - 0.82) < 0.001 and abs(test_results["precision"] - 0.807) < 0.001 and abs(test_results["recall"] - 0.829) < 0.001)
check("4. training precision 0.857, recall 0.862",
      lambda: abs(train_results["precision"] - 0.857) < 0.001 and abs(train_results["recall"] - 0.862) < 0.001)

These are the same numbers as in the LELA60331 Week 9 seminar. Compare the two pieces of code. What does the class do for you that the seminar's cells made you do by hand?

As in the seminar, the model does better on the training reviews than on the test reviews: it has **overfitted** a little.

Now try your own reviews (in lower case, with spaces around punctuation, like the Yelp data). Can you write one that fools the model?

In [ ]:
my_reviews = ["the food was amazing and the staff were friendly !",
              "terrible service , cold food . never again ."]
for review, p in zip(my_reviews, model.predict_proba(my_reviews)):
    print(round(p, 3), review)

`zip(a, b)` goes through two lists side by side, giving one item from each on every pass.

---
# Part 3 · Inspecting the weights

### 🏋️ Exercise 3 · Practice — The most important words

1. `lowest` and `highest`: the 15 words with the lowest and the highest weights. *Hint:* `top_words` returns two lists, so unpack them (Week 4).
2. `lowest_100` and `highest_100`: the same for the 100 lowest and highest
3. `not_words`: the tokens in `lowest_100` and `highest_100` that are **not** made only of letters. *Hint:* `isalpha()` (Week 2).

Look through the lists. Which words are about sentiment in general? Which suggest **overfitting**: that they tell the model about the particular reviews it was trained on (LELA60331 Week 9, Activity 8)? Are the tokens in `not_words` useful to the model, or noise?

In [ ]:
lowest, highest = ???                    # 1.
print(lowest)
print(highest)

lowest_100, highest_100 = ???           # 2.
not_words = ???   # 3.
print(not_words)

In [ ]:
# ✅ Check
check("1. lowest", lambda: lowest[:3] == ["?", "no", "not"] and len(lowest) == 15)
check("1. highest", lambda: highest[:3] == ["great", "definitely", "excellent"] and len(highest) == 15)
check("2. 100 of each", lambda: len(lowest_100) == 100 and len(highest_100) == 100 and lowest_100[:15] == lowest)
check("3. not_words", lambda: not_words == ["?", "!"])

*Write your observations here.*

---
# Part 4 · A better tokeniser: just swap the encoder

Many of the 5000 tokens in the vocabulary aren't really words: punctuation, numbers, and `n` (what is left of the line breaks in the original reviews). Exercise 3 showed that two of them, `?` and `!`, are among the most useful tokens of all. Would the model be better without the rest?

In the LELA60331 Week 9 seminar you changed the tokenisation cell and re-ran everything, losing the first model. Here, thanks to composition, you don't even need a new classifier class: write a new **encoder** (a subclass of `TextEncoder` that overrides `tokenise`, as you did with `StopwordEncoder` in Week 9), and give it to a new `TextLogisticRegression`.

### 🏋️ Exercise 4 · Practice — `CleanEncoder`

1. Complete `tokenise` in `CleanEncoder`. It should call the parent's `tokenise` and keep only the tokens that are made of letters (`isalpha()`) and are not `"n"`. (That throws away `!` and `?` too. Do you think that is a good idea? You can try other rules afterwards.)
2. Make `clean_model`: a `TextLogisticRegression` with a `CleanEncoder` of 5000 words and the same settings as `model`. Build its vocabulary from all the texts, and train it.
3. `clean_results`: its results on the test set. Is it better or worse than `model`? Look at its top words: what has changed? Can you explain the difference? (Think about Exercise 3.) Try an encoder that keeps `!` and `?`.

In [ ]:
class CleanEncoder(TextEncoder):
    """A TextEncoder that only uses tokens made of letters."""

    def tokenise(self, text):
        return ???    # 1.

print(CleanEncoder().tokenise("i paid $ 20 for 2 pizzas ! n nthe best ."))

clean_model = ???   # 2.
???
???

clean_results = ???                   # 3.
print("naive tokens:", test_results)
print("clean tokens:", clean_results)
print(clean_model.top_words(15))

In [ ]:
# ✅ Check
check("1. tokenise", lambda: CleanEncoder().tokenise("i paid $ 20 for 2 pizzas ! n nthe best .") == ["i", "paid", "for", "pizzas", "nthe", "best"])
check("2. a TextLogisticRegression with a CleanEncoder", lambda: isinstance(clean_model, TextLogisticRegression) and isinstance(clean_model._encoder, CleanEncoder))
check("2. no punctuation in its vocabulary", lambda: clean_model.one_hot_encode([". ! ?"]) == [[]])
check("3. clean_results", lambda: set(clean_results) == {"accuracy", "precision", "recall"})
check("the first model is unchanged", lambda: model.one_hot_encode(["."]) == [[0]] and abs(model.evaluate(test_texts, test_labels)["accuracy"] - 0.82) < 0.001)

Look at the token `nthe` in the output of `tokenise`: it is a line break stuck to the next word. ⭐ **Challenge (optional):** write an encoder that also fixes tokens like this, by removing an `n` from the start of any token that isn't a word on its own. How can you tell which tokens to fix? Compare its results with the other two models.

In [ ]:
# ⭐ Challenge

---
## ✏️ Summary

You can now:

* tell an **"is a"** relationship (inheritance) from a **"has a"** relationship (composition), and use both
* write a subclass that overrides a few small methods and reuses everything else, including methods it never mentions
* use `super()` to add to a parent's method, or to hand it a different form of input
* swap one part of a program (the encoder) for another without changing the rest
* train, evaluate and inspect a text classifier with a handful of method calls

Over Weeks 8, 9 and 10 you have written, in about 150 lines, the complete review classifier from the LELA60331 seminars. Libraries such as scikit-learn are organised in exactly this way: a model is an object with methods like `fit` and `predict`, and text is turned into features by separate "vectorizer" objects.